## Expert Knowledge Worker

### A question answering agent that is an expert knowledge worker
### To be used by employees of Insurellm, an Insurance Tech company
### The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

In [ ]:
# imports

import os
import glob
from dotenv import load_dotenv
import gradio as gr

# Using Langchain
Here we are importing two packages from langchain , document_loaders which is a utility class that helps us to load files.
DirectoryLoader is a class that loads complete directory
TextLoader class loads an individual text file.

Then we are importing the class CharacterTextSplitter .
This class takes up a document and divides it into chunks of document

In [ ]:
# imports for langchain

from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import CharacterTextSplitter

In [ ]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-4o-mini"
db_name = "vector_db"

In [ ]:
# Load environment variables in a file called .env

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')

## Reading the documents using langchain's loaders

In [ ]:
# Read in documents using LangChain's loaders
# Take everything in all the sub-folders of our knowledgebase

# 1. Fist we get the list of different folders in out knowledgbase
folders = glob.glob("knowledge-base/*")
print(folders)  # 'knowledge-base/products', 'knowledge-base/contracts', 'knowledge-base/company', 'knowledge-base/employees']

# 2. Define the encoding format to be used while loading the documents from the files 
text_loader_kwargs = {'encoding': 'utf-8'}

documents = []
# 3. For each of the folders
for folder in folders:
    
    # 4. Load the name of the folder
    doc_type = os.path.basename(folder)

    # 5. Load the contents of the files. Each .md file will be read , the text will be loaded using the class TextLoader 
    # and the encoding used will be text_loader_kwargs
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs=text_loader_kwargs)

    # 6. Loads the document
    folder_docs = loader.load()

    # 7. For each of the document , set the doc type as the name of the folder like company , employee etc.
    # This is for setting metadata for the document
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(documents)

Output -
```print(doc_type) ```

```
# products
# contracts
# company
# employees
```

In [ ]:
len(documents)

In [ ]:
documents[24]

'''
The structre of each elements in the list is -
{
metadata={'source': 'knowledge-base/employees/Maxine Thompson.md', 'doc_type': 'employees'}, 
page_content="# HR Record\n\n# Maxine Thompson\n\n## Summary\n- **Date of Birth:** January 15, 1991  \n- **Job Title:** Data Engineer ....
}

the metadata dictionary was also created by the Langchain DocumentLoader , we just added another key doc_type to it using our code.
'''

## Below we have the text_splitter
This code will take each documents and split the documents into chunks of the size we have passed. The parameters passed are -

1. **chunk_size** which is roughly how many characters do we want to fit in each chunk.
       Roughly because we are going to give langchain some discretion to make sure it tries to split these chunks within sensible boundaries , like not cutting in   the middle of a paragraph or word etc , and take the meaningful sequence even if 1000 characters has surpassed and so on.

2. **chunk_overlap** says we do not want these chunks of characters to be completely seperate from each other. There has to be some level of overlap between them. So there should be like some common content between the two chunks.

This is important because suppose we have a query , we should be able to pluck out all the chunks that have relevant data for the query. We do not want to risk that a critical word gets included only in one chunk and there is no other relation of this chunk with other chunks , but other chunks may have some data that is relevant to the query.

In [ ]:
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

In [ ]:
len(chunks)

In [ ]:
chunks[6]

'''
Document(
metadata={'source': 'knowledge-base/products/Markellm.md', 'doc_type': 'products'}, 
page_content='- **User-Friendly Interface**: Designed with user experience in mind, Markellm features an intuitive interface that allows consumers to easily browse and compare various insurance offerings from multiple providers.\n\n- **Real-Time Quotes**: Consumers can receive real-time quotes from different insurance companies, empowering them to make informed decisions quickly without endless back-and-forth communication.\n\n- **Customized Recommendations**: Based on user profiles and preferences, Markellm provides personalized insurance recommendations, ensuring consumers find the right coverage at competitive rates.\n\n- **Secure Transactions**: Markellm prioritizes security, employing robust encryption methods to ensure that all transactions and data exchanges are safe and secure.\n\n- **Customer Support**: Our dedicated support team is always available to assist both consumers and insurers throughout the process, providing guidance and answering any questions that may arise.'
)
'''

In [ ]:
doc_types = set(chunk.metadata['doc_type'] for chunk in chunks)
print(f"Document types found: {', '.join(doc_types)}")

#### Lets do an experiment wherein we will look through each of the different chunks and check which chunk has some keyword in it.

In [ ]:
for chunk in chunks:
    if 'CEO' in chunk.page_content:
        print(chunk)
        print("_________")